# Tidyverse দিয়ে ডেটা র‍্যাংগলিং ও প্রক্রিয়াকরণ

বহুভাষিক ডেটা পাইপলাইন: Python ডেটা ডাউনলোড করে → R তা dplyr দিয়ে প্রসেস করে → Python ফলাফল ভিজ্যুয়ালাইজ করে।

**SharedVFS** প্রদর্শন করে — যা একটি শেয়ার করা ফাইল সিস্টেম এবং Python ও R-কে ফাইল আদান-প্রদান করতে দেয়।

## ১. Python: ডেটাসেট ডাউনলোড করুন

In [ ]:
import micropip
await micropip.install('pandas')
import pandas as pd, pyodide.http, os

url = "https://raw.githubusercontent.com/resbaz/r-novice-gapminder-files/master/data/gapminder-FiveYearData.csv"
resp = await pyodide.http.pyfetch(url)
text = await resp.string()

os.makedirs("/shared/data", exist_ok=True)
with open("/shared/data/gapminder.csv", "w") as f:
    f.write(text)

df = pd.read_csv("/shared/data/gapminder.csv")
print(f"ডাউনলোড করা হয়েছে: {df.shape[0]} টি সারি, {df.shape[1]} টি কলাম")
df.head()

## ২. R: dplyr + tidyr ইনস্টল করুন এবং শেয়ার করা ডেটা পড়ুন

In [ ]:
install.packages(c("dplyr", "tidyr"))
library(dplyr)

gap <- read.csv("/shared/data/gapminder.csv")
cat("SharedVFS থেকে পড়া হয়েছে:", nrow(gap), "সারি\n")
glimpse(gap)

## ৩. dplyr: মহাদেশ অনুযায়ী সারসংক্ষেপ (2007)

In [ ]:
gap %>%
  filter(year == 2007) %>%
  group_by(continent) %>%
  summarize(
    countries = n(),
    mean_life = round(mean(lifeExp), 1),
    median_gdp = round(median(gdpPercap), 0),
    total_pop = sum(as.numeric(pop))
  ) %>%
  arrange(desc(mean_life))

## ৪. dplyr: গড় আয়ুতে সবচেয়ে বড় বৃদ্ধি

In [ ]:
gains <- gap %>%
  filter(year %in% c(1952, 2007)) %>%
  select(country, continent, year, lifeExp) %>%
  tidyr::pivot_wider(names_from = year, values_from = lifeExp,
                     names_prefix = "y") %>%
  mutate(gain = y2007 - y1952) %>%
  arrange(desc(gain)) %>%
  head(10)
gains

## ৫. dplyr: মহাদেশ অনুযায়ী জনসংখ্যা বৃদ্ধি

In [ ]:
pop_growth <- gap %>%
  filter(year %in% c(1952, 2007)) %>%
  group_by(continent, year) %>%
  summarize(total_pop = sum(as.numeric(pop)), .groups = "drop") %>%
  tidyr::pivot_wider(names_from = year, values_from = total_pop,
                     names_prefix = "pop_") %>%
  mutate(growth_pct = round((pop_2007 / pop_1952 - 1) * 100, 1)) %>%
  arrange(desc(growth_pct))
pop_growth

## ৬. R: ফলাফল SharedVFS-এ লিখুন

In [ ]:
# Python-এ ভিজ্যুয়ালাইজ করার জন্য মহাদেশের সারসংক্ষেপ লিখুন
summary_2007 <- gap %>%
  filter(year == 2007) %>%
  group_by(continent) %>%
  summarize(
    mean_life = round(mean(lifeExp), 1),
    mean_gdp = round(mean(gdpPercap), 0),
    .groups = "drop"
  )
write.csv(summary_2007, "/shared/data/r_summary.csv", row.names = FALSE)
cat("/shared/data/r_summary.csv ফাইলটি লেখা হয়েছে\n")
summary_2007

## ৭. Python: R-এর ফলাফল ভিজ্যুয়ালাইজ করুন

In [ ]:
import micropip
await micropip.install('plotly')
import plotly.express as px
import json, js
from plotly.utils import PlotlyJSONEncoder

def plain_plotly(value):
    if hasattr(value, 'tolist'):
        return value.tolist()
    if isinstance(value, dict):
        return {key: plain_plotly(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [plain_plotly(item) for item in value]
    return value

def show_plotly(fig):
    payload = plain_plotly(fig.to_plotly_json())
    js.renderPlot(json.dumps({"traces": payload["data"], "layout": payload["layout"]}, cls=PlotlyJSONEncoder))

r_summary = pd.read_csv("/shared/data/r_summary.csv")
print("SharedVFS থেকে পড়া হয়েছে (R দ্বারা লিখিত):")
print(r_summary.to_string(index=False))

fig = px.bar(r_summary, x="continent", y="mean_life",
             title="মহাদেশ অনুযায়ী গড় আয়ু (২০০৭) — R → Python থেকে",
             labels={"mean_life": "গড় আয়ু (বছর)", "continent": "মহাদেশ"},
             color="continent")
fig.update_layout(template='plotly_dark', showlegend=False)
show_plotly(fig)

In [ ]:
fig = px.scatter(r_summary, x="mean_gdp", y="mean_life",
                 text="continent", size=[40]*len(r_summary),
                 title="মহাদেশ অনুযায়ী GDP বনাম গড় আয়ু (R সারসংক্ষেপ → Python প্লট)",
                 labels={"mean_gdp": "মাথাপিছু গড় GDP", "mean_life": "গড় আয়ু"})
fig.update_traces(textposition="top center")
fig.update_layout(template='plotly_dark')
fig.update_yaxes(range=[r_summary['mean_life'].min() - 2, r_summary['mean_life'].max() + 6])
show_plotly(fig)

## মূল শিক্ষা

- **Python** CSV ডেটা `/shared/data/`-তে ডাউনলোড করেছে
- **R** তা SharedVFS-এর মাধ্যমে পড়েছে এবং dplyr পাইপলাইন দিয়ে প্রসেস করেছে
- **R** সারসংক্ষেপটি পুনরায় `/shared/data/r_summary.csv`-তে লিখেছে
- **Python** R-এর আউটপুট পড়েছে এবং Plotly দিয়ে ইন্টারেক্টিভ চার্ট তৈরি করেছে

সমস্ত ফাইল শেয়ারিং SharedVFS-এর মাধ্যমে ঘটে — কোনো ম্যানুয়াল ইমপোর্ট বা এক্সপোর্টের প্রয়োজন নেই।